# Pemodelan Prediksi Harga Pangan
**Skripsi Sains Data — Random Forest Regressor (Parameter Final)**

Notebook ini melakukan:
- Load dataset `features_all_dataset.csv`
- Split kronologis 80:20 (tanpa random shuffle)
- Melatih model dengan parameter **final** (sudah diketahui dari 2-Tahap GridSearch + TimeSeriesSplit)
- Evaluasi global pada test set
- Menyimpan semua artefak ke `models/` dan `results/`

> **Catatan**: Notebook ini TIDAK melakukan tuning ulang dan TIDAK menampilkan evaluasi per komoditas.
> Semua evaluasi lanjutan dilakukan di `evaluation_model.ipynb` menggunakan `test_predictions.csv`.

## SEL 1 — Import, Konstanta PATH, Fungsi Bantu

In [1]:
# =============================================================================
# SEL 1 — Import semua library, definisi PATH, BEST_PARAMS, dan fungsi bantu
# =============================================================================

import os
import json
import pickle
import datetime

import numpy  as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------------------------
# PATH KONSTANTA — semua path relatif terhadap root proyek (satu level di atas notebook/)
# ---------------------------------------------------------------------------
DATA_PATH    = "../processed_data/features/features_all_dataset.csv"
MODEL_DIR    = "../models"
RESULTS_DIR  = "../results"

# Buat folder jika belum ada
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"MODEL_DIR   : {os.path.abspath(MODEL_DIR)}")
print(f"RESULTS_DIR : {os.path.abspath(RESULTS_DIR)}")

# ---------------------------------------------------------------------------
# BEST_PARAMS — parameter final dari 2-Tahap GridSearch + TimeSeriesSplit
# Tidak perlu tuning ulang; nilai ini bersifat FINAL.
# Best CV MAE : Rp 1,107.96
# ---------------------------------------------------------------------------
BEST_PARAMS = {
    "max_depth"         : 10,
    "min_samples_leaf"  : 5,
    "min_samples_split" : 12,
    "n_estimators"      : 100,
}
BEST_CV_MAE = 1107.96   # Rp — dari proses GridSearch sebelumnya

print(f"\nBEST_PARAMS : {BEST_PARAMS}")
print(f"Best CV MAE : Rp {BEST_CV_MAE:,.2f}")

# ---------------------------------------------------------------------------
# FUNGSI BANTU — dipakai konsisten di modelling.ipynb DAN evaluation_model.ipynb
# ---------------------------------------------------------------------------

def mape_fn(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean Absolute Percentage Error (MAPE) — hindari division-by-zero.

    Parameters
    ----------
    y_true : array-like, nilai aktual
    y_pred : array-like, nilai prediksi

    Returns
    -------
    float : MAPE dalam persen (%)
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask   = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def eval_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Hitung MAE, RMSE, MAPE, dan R² secara konsisten.

    Parameters
    ----------
    y_true : array-like, nilai aktual
    y_pred : array-like, nilai prediksi

    Returns
    -------
    dict : {"mae", "rmse", "mape", "r2"}
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae"  : float(mean_absolute_error(y_true, y_pred)),
        "rmse" : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape" : mape_fn(y_true, y_pred),
        "r2"   : float(r2_score(y_true, y_pred)),
    }


print("\n✅ Library, konstanta, dan fungsi bantu berhasil didefinisikan.")

MODEL_DIR   : c:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\models
RESULTS_DIR : c:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\results

BEST_PARAMS : {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 12, 'n_estimators': 100}
Best CV MAE : Rp 1,107.96

✅ Library, konstanta, dan fungsi bantu berhasil didefinisikan.


## SEL 2 — Load Dataset & Validasi

In [2]:
# =============================================================================
# SEL 2 — Load features_all_dataset.csv, sort kronologis, validasi kolom & NaN
# =============================================================================

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset dimuat  : {df.shape[0]:,} baris × {df.shape[1]} kolom")

# Parse dan sort tanggal ascending (wajib untuk split kronologis)
df["Tanggal"] = pd.to_datetime(df["Tanggal"])
df = df.sort_values("Tanggal").reset_index(drop=True)
print(f"Rentang tanggal : {df['Tanggal'].min().date()} → {df['Tanggal'].max().date()}")

# ---------------------------------------------------------------------------
# Validasi semua kolom wajib tersedia
# ---------------------------------------------------------------------------
REQUIRED_COLS = [
    # Kolom identitas (untuk groupby / evaluasi)
    "Tanggal", "Komoditas",
    # Target
    "Harga",
    # 13 fitur model
    "Tahun", "Bulan", "Hari", "DayOfWeek", "Quarter", "WeekOfYear",
    "Harga_Kemarin", "Harga_Minggu_Lalu",
    "Rolling_Mean_7", "Rolling_Mean_14", "Rolling_Std_7",
    "Rolling_Max_7", "Rolling_Min_7",
    # Catatan: kolom 'Sumber' SENGAJA tidak diperlukan (55.98% missing — temuan EDA)
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing_cols, f"Kolom wajib tidak ditemukan: {missing_cols}"
print("\n✅ Semua kolom wajib tersedia.")

# ---------------------------------------------------------------------------
# Cek dan print jumlah NaN per kolom
# ---------------------------------------------------------------------------
nan_counts = df[REQUIRED_COLS].isnull().sum()
nan_cols   = nan_counts[nan_counts > 0]

print("\nJumlah NaN per kolom (hanya kolom wajib):")
if nan_cols.empty:
    print("  → Tidak ada NaN pada kolom wajib. ✅")
else:
    print(nan_cols.to_string())

# Pastikan fitur dan target bebas NaN sebelum training
FEATURE_AND_TARGET_COLS = [
    "Harga",
    "Tahun", "Bulan", "Hari", "DayOfWeek", "Quarter", "WeekOfYear",
    "Harga_Kemarin", "Harga_Minggu_Lalu",
    "Rolling_Mean_7", "Rolling_Mean_14", "Rolling_Std_7",
    "Rolling_Max_7", "Rolling_Min_7",
]
total_nan = df[FEATURE_AND_TARGET_COLS].isnull().sum().sum()
assert total_nan == 0, (
    f"Masih ada {total_nan} NaN pada kolom fitur/target! "
    "Periksa ulang pipeline feature engineering."
)
print("\n✅ Fitur dan target bebas NaN — siap untuk training.")

Dataset dimuat  : 76,363 baris × 17 kolom
Rentang tanggal : 2021-01-18 → 2026-11-10

✅ Semua kolom wajib tersedia.

Jumlah NaN per kolom (hanya kolom wajib):
  → Tidak ada NaN pada kolom wajib. ✅

✅ Fitur dan target bebas NaN — siap untuk training.


## SEL 3 — Definisi feature_cols & Split Kronologis 80:20

In [3]:
# =============================================================================
# SEL 3 — Definisi feature_cols dan split kronologis 80:20 berdasarkan tanggal
# =============================================================================
#
# CATATAN PENTING — MENGAPA TIDAK PAKAI iloc:
# Dataset berisi beberapa komoditas per tanggal (15–57 komoditas/hari).
# Jumlah komoditas bertambah dari waktu ke waktu (2021: 15, 2026: 57).
# Split dengan iloc[:n_train] dapat memotong di tengah sebuah tanggal,
# sehingga tanggal yang sama muncul di train DAN test → assert leakage gagal.
# Solusi: tentukan cutoff_date dari akumulasi baris per tanggal, lalu
# filter dengan (Tanggal <= cutoff_date) dan (Tanggal > cutoff_date).
# Ini menjamin setiap tanggal hanya ada di satu split.

# ---------------------------------------------------------------------------
# Definisi feature_cols — 13 fitur model
# Kolom yang DIKELUARKAN (dokumentasi):
#   - Sumber    : 55.98% missing values (temuan EDA) → tidak informatif
#   - Tanggal   : hanya digunakan untuk sorting kronologis
#   - Komoditas : dipakai untuk groupby evaluasi di evaluation_model.ipynb
# ---------------------------------------------------------------------------
feature_cols = [
    # Fitur kalender
    "Tahun", "Bulan", "Hari", "DayOfWeek", "Quarter", "WeekOfYear",
    # Fitur lag harga
    "Harga_Kemarin", "Harga_Minggu_Lalu",
    # Fitur rolling window
    "Rolling_Mean_7", "Rolling_Mean_14", "Rolling_Std_7",
    "Rolling_Max_7", "Rolling_Min_7",
]
assert len(feature_cols) == 13, f"Expected 13 fitur, got {len(feature_cols)}"
print(f"feature_cols ({len(feature_cols)} fitur) : {feature_cols}")

# ---------------------------------------------------------------------------
# Tentukan cutoff_date berdasarkan akumulasi baris per tanggal unik.
# Cari tanggal terakhir di mana cumulative baris masih <= target 80% baris.
# Ini memastikan proporsi baris ≈ 80:20 dan tidak ada overlap tanggal.
# ---------------------------------------------------------------------------
n_total        = len(df)
target_n_train = int(n_total * 0.80)

# Akumulasi baris per tanggal (df sudah di-sort ascending sejak SEL 2)
cum_per_date = (
    df.groupby("Tanggal")
    .size()
    .cumsum()
    .reset_index(name="cum_rows")
)

# Cutoff: tanggal terakhir yang akumulasinya masih <= target baris train
cutoff_date = cum_per_date.loc[
    cum_per_date["cum_rows"] <= target_n_train, "Tanggal"
].max()

# Split berdasarkan tanggal — setiap tanggal hanya ada di satu split
train_df = df[df["Tanggal"] <= cutoff_date].copy()   # lengkap: Tanggal, Komoditas, Harga
test_df  = df[df["Tanggal"] >  cutoff_date].copy()   # lengkap: Tanggal, Komoditas, Harga

n_train  = len(train_df)
n_test   = len(test_df)

# Buat X dan y
X_train = train_df[feature_cols]
y_train = train_df["Harga"]
X_test  = test_df[feature_cols]
y_test  = test_df["Harga"]

# ---------------------------------------------------------------------------
# Assert guard — pastikan tidak ada overlap tanggal (anti data-leakage)
# ---------------------------------------------------------------------------
assert test_df["Tanggal"].min() > train_df["Tanggal"].max(), (
    "LEAKAGE TERDETEKSI: tanggal test set overlap dengan train set!"
)

# Informasi split
train_pct = n_train / n_total * 100
test_pct  = n_test  / n_total * 100

print(f"\nTotal baris    : {n_total:,}")
print(f"Train          : {n_train:,} baris ({train_pct:.2f}%)  "
      f"| {train_df['Tanggal'].min().date()} → {train_df['Tanggal'].max().date()}")
print(f"Test           : {n_test:,}  baris ({test_pct:.2f}%)  "
      f"| {test_df['Tanggal'].min().date()} → {test_df['Tanggal'].max().date()}")
print(f"Cutoff tanggal : {cutoff_date.date()}")
print("\n✅ Split kronologis tanpa overlap — aman dari data leakage.")

feature_cols (13 fitur) : ['Tahun', 'Bulan', 'Hari', 'DayOfWeek', 'Quarter', 'WeekOfYear', 'Harga_Kemarin', 'Harga_Minggu_Lalu', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Std_7', 'Rolling_Max_7', 'Rolling_Min_7']

Total baris    : 76,363
Train          : 61,087 baris (80.00%)  | 2021-01-18 → 2026-02-15
Test           : 15,276  baris (20.00%)  | 2026-02-16 → 2026-11-10
Cutoff tanggal : 2026-02-15

✅ Split kronologis tanpa overlap — aman dari data leakage.


## SEL 4 — Inisialisasi & Training Model Final

In [4]:
# =============================================================================
# SEL 4 — Inisialisasi RandomForestRegressor dengan BEST_PARAMS dan latih model
# TIDAK ADA TUNING DI SINI — parameter sudah final dari 2-Tahap GridSearch
# =============================================================================

best_model = RandomForestRegressor(
    max_depth         = BEST_PARAMS["max_depth"],
    min_samples_leaf  = BEST_PARAMS["min_samples_leaf"],
    min_samples_split = BEST_PARAMS["min_samples_split"],
    n_estimators      = BEST_PARAMS["n_estimators"],
    random_state      = 42,
    n_jobs            = -1,
)

print("Parameter model yang digunakan:")
for k, v in BEST_PARAMS.items():
    print(f"  {k:<22}: {v}")
print(f"  {'random_state':<22}: 42")
print(f"  {'n_jobs':<22}: -1 (semua CPU)")
print(f"\nBest CV MAE (referensi GridSearch) : Rp {BEST_CV_MAE:,.2f}")

print(f"\nMelatih model pada {len(X_train):,} baris data latih ...")
best_model.fit(X_train, y_train)
print("✅ Training selesai.")

Parameter model yang digunakan:
  max_depth             : 10
  min_samples_leaf      : 5
  min_samples_split     : 12
  n_estimators          : 100
  random_state          : 42
  n_jobs                : -1 (semua CPU)

Best CV MAE (referensi GridSearch) : Rp 1,107.96

Melatih model pada 61,087 baris data latih ...


✅ Training selesai.


## SEL 5 — Prediksi & Evaluasi Global

In [5]:
# =============================================================================
# SEL 5 — Prediksi y_pred_global pada X_test, hitung & print metrik global
# =============================================================================

# Prediksi
y_pred_global = best_model.predict(X_test)

# Hitung metrik
metrics_global = eval_metrics(y_test.values, y_pred_global)

print("=" * 60)
print("EVALUASI GLOBAL — Seluruh Test Set")
print("=" * 60)
print(f"Jumlah baris test : {len(y_test):,}")
print(f"MAE               : Rp {metrics_global['mae']:>12,.2f}")
print(f"RMSE              : Rp {metrics_global['rmse']:>12,.2f}")
print(f"MAPE              : {metrics_global['mape']:>13.4f} %")
print(f"R²                : {metrics_global['r2']:>15.6f}")
print()
print("⚠️  CATATAN PENTING:")
print("   ~98% baris test set merupakan data forward-fill (ffill) dari hari")
print("   sebelumnya — bukan harga transaksi riil. Metrik global di atas")
print("   SANGAT dipengaruhi oleh dominasi baris ffill tersebut sehingga")
print("   terlihat sangat baik (MAE rendah, R² mendekati 1).")
print("   Evaluasi yang lebih representatif (per komoditas, hari riil vs ffill)")
print("   dilakukan di evaluation_model.ipynb menggunakan test_predictions.csv.")
print("=" * 60)

EVALUASI GLOBAL — Seluruh Test Set
Jumlah baris test : 15,276
MAE               : Rp       227.64
RMSE              : Rp     1,027.27
MAPE              :        0.5583 %
R²                :        0.999800

⚠️  CATATAN PENTING:
   ~98% baris test set merupakan data forward-fill (ffill) dari hari
   sebelumnya — bukan harga transaksi riil. Metrik global di atas
   SANGAT dipengaruhi oleh dominasi baris ffill tersebut sehingga
   terlihat sangat baik (MAE rendah, R² mendekati 1).
   Evaluasi yang lebih representatif (per komoditas, hari riil vs ffill)
   dilakukan di evaluation_model.ipynb menggunakan test_predictions.csv.


## SEL 6 — Simpan Semua Artefak

In [6]:
# =============================================================================
# SEL 6 — Simpan semua artefak ke models/ dan results/
# File ini adalah JEMBATAN ke evaluation_model.ipynb
# =============================================================================

# ---------------------------------------------------------------------------
# 1. model_rf.pkl — model binary (pickle)
# ---------------------------------------------------------------------------
model_path = os.path.join(MODEL_DIR, "model_rf.pkl")
with open(model_path, "wb") as f:
    pickle.dump(best_model, f)
print(f"✅ Disimpan : {model_path}")

# ---------------------------------------------------------------------------
# 2. feature_columns.json — urutan feature_cols (wajib sama di kedua notebook)
# ---------------------------------------------------------------------------
feat_path = os.path.join(MODEL_DIR, "feature_columns.json")
with open(feat_path, "w") as f:
    json.dump(feature_cols, f, indent=2)
print(f"✅ Disimpan : {feat_path}")

# ---------------------------------------------------------------------------
# 3. split_info.json — informasi cutoff, tanggal, n_train, n_test
# ---------------------------------------------------------------------------
split_info = {
    "cutoff_date"      : cutoff_date.strftime("%Y-%m-%d"),
    "train_start"      : train_df["Tanggal"].min().strftime("%Y-%m-%d"),
    "train_end"        : train_df["Tanggal"].max().strftime("%Y-%m-%d"),
    "test_start"       : test_df["Tanggal"].min().strftime("%Y-%m-%d"),
    "test_end"         : test_df["Tanggal"].max().strftime("%Y-%m-%d"),
    "n_total"          : int(n_total),
    "n_train"          : int(n_train),
    "n_test"           : int(n_test),
    "train_pct"        : round(train_pct, 4),
    "test_pct"         : round(test_pct,  4),
    "split_method"     : "chronological_iloc_80_20",
}
split_path = os.path.join(MODEL_DIR, "split_info.json")
with open(split_path, "w") as f:
    json.dump(split_info, f, indent=2, ensure_ascii=False)
print(f"✅ Disimpan : {split_path}")

# ---------------------------------------------------------------------------
# 4. training_metadata.json — metadata lengkap
# ---------------------------------------------------------------------------
training_metadata = {
    "created_at"         : datetime.datetime.now().isoformat(),
    "dataset_path"       : DATA_PATH,
    "n_total_rows"       : int(n_total),
    "n_features"         : len(feature_cols),
    "feature_cols"       : feature_cols,
    # Kolom yang dikeluarkan beserta alasannya (temuan EDA)
    "excluded_columns"   : {
        "Sumber"    : "55.98% missing values — tidak informatif untuk model",
        "Tanggal"   : "digunakan hanya untuk sorting kronologis",
        "Komoditas" : "digunakan untuk groupby evaluasi di evaluation_model.ipynb",
    },
    "split_info"         : split_info,
    "model_class"        : "RandomForestRegressor",
    "model_library"      : "scikit-learn",
    "best_params"        : BEST_PARAMS,
    "random_state"       : 42,
    "tuning_method"      : "2-Tahap GridSearch + TimeSeriesSplit",
    "best_cv_mae"        : BEST_CV_MAE,
    "cv_metric"          : "neg_mean_absolute_error",
    "global_test_metrics": metrics_global,
    "note_on_metrics"    : (
        "~98% test set adalah baris forward-fill (ffill). "
        "Metrik global sangat dipengaruhi dominasi ffill. "
        "Evaluasi genuine (hari riil & per komoditas) ada di evaluation_model.ipynb."
    ),
}
metadata_path = os.path.join(MODEL_DIR, "training_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False)
print(f"✅ Disimpan : {metadata_path}")

# ---------------------------------------------------------------------------
# 5. test_predictions.csv — JEMBATAN ke evaluation_model.ipynb
#    Berisi: semua kolom test_df + y_pred (prediksi) + residual (y_true - y_pred)
# ---------------------------------------------------------------------------
test_pred_df = test_df.copy()
test_pred_df["y_pred"]   = y_pred_global
test_pred_df["residual"] = test_pred_df["Harga"] - test_pred_df["y_pred"]

pred_path = os.path.join(RESULTS_DIR, "test_predictions.csv")
test_pred_df.to_csv(pred_path, index=False)
print(f"✅ Disimpan : {pred_path}")
print(f"   (shape: {test_pred_df.shape[0]:,} baris × {test_pred_df.shape[1]} kolom)")
print(f"   Kolom baru: y_pred, residual")

# ---------------------------------------------------------------------------
# Ringkasan akhir
# ---------------------------------------------------------------------------
print()
print("=" * 60)
print("SEMUA ARTEFAK BERHASIL DISIMPAN")
print("=" * 60)
print(f"  models/model_rf.pkl           → model biner")
print(f"  models/feature_columns.json   → urutan 13 fitur")
print(f"  models/split_info.json        → info cutoff & ukuran split")
print(f"  models/training_metadata.json → metadata lengkap")
print(f"  results/test_predictions.csv  → jembatan ke evaluation_model.ipynb")
print("=" * 60)

✅ Disimpan : ../models\model_rf.pkl
✅ Disimpan : ../models\feature_columns.json
✅ Disimpan : ../models\split_info.json
✅ Disimpan : ../models\training_metadata.json
✅ Disimpan : ../results\test_predictions.csv
   (shape: 15,276 baris × 19 kolom)
   Kolom baru: y_pred, residual

SEMUA ARTEFAK BERHASIL DISIMPAN
  models/model_rf.pkl           → model biner
  models/feature_columns.json   → urutan 13 fitur
  models/split_info.json        → info cutoff & ukuran split
  models/training_metadata.json → metadata lengkap
  results/test_predictions.csv  → jembatan ke evaluation_model.ipynb
